# 🐍 Master Guide: High-Level Concurrency (`concurrent.futures`)
> **Single-Source Revision Guide**: Covers `ThreadPoolExecutor`, `ProcessPoolExecutor`, `Future` objects, `as_completed()`, timeouts, and graceful cancellations.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
The `concurrent.futures` module provides a clean, unified high-level abstraction over both threads and processes using **Executors** and **Futures**.

---
## 🧠 2. CPython Architecture: ThreadPool vs ProcessPool

```
                 concurrent.futures.Executor (Interface)
                                │
            ┌───────────────────┴───────────────────┐
            ▼                                       ▼
  ThreadPoolExecutor                      ProcessPoolExecutor
  - Backed by OS threads                  - Backed by separate Python processes
  - Subject to GIL                        - Bypasses GIL (True multicore)
  - Shared memory                         - Separate memory (IPC via pickle)
  - Best for I/O tasks                    - Best for CPU-heavy tasks
```

### What is a `Future`?
A `Future` is an eventual result of an asynchronous computation. It provides methods like `.done()`, `.result()`, and `.cancel()`.

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: Mapping Tasks with `ThreadPoolExecutor`

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def fetch_url(url):
    time.sleep(0.1)  # Simulating network latency
    return f"Status 200 from {url}"

urls = [
    "https://api.github.com",
    "https://api.python.org",
    "https://api.google.com"
]

with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(fetch_url, urls)
    for r in results:
        print(r)

---
## 🟡 4. Part 2: Intermediate Features

### Submitting Tasks & Processing in Completion Order (`as_completed`)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import random

def simulated_work(task_id):
    delay = random.uniform(0.05, 0.2)
    time.sleep(delay)
    return f"Task {task_id} completed in {delay*1000:.1f} ms"

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(simulated_work, i) for i in range(1, 6)]
    
    # as_completed yields futures as soon as they finish!
    print("Results as they complete (non-deterministic order):")
    for future in as_completed(futures):
        print(" ->", future.result())

---
## 📋 5. Part 3: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Concept                            | Key Rule                              |
+------------------------------------+---------------------------------------+
| ThreadPoolExecutor                 | I/O tasks, shared memory              |
| ProcessPoolExecutor                | CPU tasks, multicore parallelism      |
| executor.map(func, items)          | Yields results in original input order|
| executor.submit(func, *args)       | Returns a single Future object        |
| as_completed(futures)              | Yields futures as soon as they finish |
| future.result(timeout=10)          | Blocks until finished or raises err   |
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. High-Level ThreadPool & ProcessPool Execution
Using `concurrent.futures.ThreadPoolExecutor` to handle concurrent tasks cleanly with return values and exception propagation.

In [ ]:
# Concurrent Futures Execution Pipeline
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def fetch_exchange_rate(currency):
    """Simulates fetching live currency exchange rates."""
    time.sleep(0.02)
    rates = {"USD": 83.25, "EUR": 89.50, "GBP": 105.10, "JPY": 0.55}
    return currency, rates.get(currency, 0.0)

currencies = ["USD", "EUR", "GBP", "JPY"]

print("--- Fetching Rates via ThreadPoolExecutor ---")
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(fetch_exchange_rate, c) for c in currencies]
    for future in as_completed(futures):
        curr, rate = future.result()
        print(f"  1 {curr} = Rs. {rate:.2f}")